## ToolStrategy是最通用的结构化输出策略
#### 这里又有5种方式： Pydantic，TypeDict，JsonSchema，@dataclass和Union[type1,type2] 这里的type1和type2是同一种方式的2个不同对象


### 1.Pydantic
#### 举例1

In [9]:
from typing import Literal

from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from langchain.agents.structured_output import AutoStrategy
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from rich import print as rprint
# 1.初始化模型
model = ChatOpenAI(
    model="carstenuhlig/omnicoder-9b:latest",
    # model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)
# 2.使用Pydantic结构方式来定义一个类
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱")
    phone: str = Field(description="用户电话")

# 3.创建agent
agent = create_agent(
    model=model,
    tools=[],
    response_format=AutoStrategy(schema=ContactInfo), # 结构化输出的第3种方式
    system_prompt="Agent的行为指令" # 可选
)
# 3.调用
result = agent.invoke({
    "messages":[{"role":"user","content":"请提取项目文本的用户信息：小李的email是 wxm1234@gmail.com,电话是13532677677"}]
})

rprint(result)






{
    'messages': [
        HumanMessage(
            content='请提取项目文本的用户信息：小李的email是 wxm1234@gmail.com,电话是13532677677',
            additional_kwargs={},
            response_metadata={},
            id='4616422c-67cf-4dd4-89c6-ef0e5312cd24'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 133,
                    'prompt_tokens': 199,
                    'total_tokens': 332,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'carstenuhlig/omnicoder-9b:latest',
                'system_fingerprint': 'fp_ollama',
                'id': 'chatcmpl-61',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a104be-6381-74a1-989c-2bba41d31b8d-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小李', 'email': 'wxm1234@gmail.com', 'phone': '13532677677'},
                    'id': 'call_jnj4ph10',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 199,
                'output_tokens': 133,
                'total_tokens': 332,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小李' email='wxm1234@gmail.com' phone='13532677677'",
            name='ContactInfo',
            id='e4ac4689-d12e-47b8-b945-cb3e84b51eab',
            tool_call_id='call_jnj4ph10'
        )
    ],
    'structured_response': ContactInfo(name='小李', email='wxm1234@gmail.com', phone='13532677677')
}

#### Pydantic举例2

In [10]:
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from langchain.agents.structured_output import AutoStrategy, ToolStrategy
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from rich import print as rprint
from typing import Literal

## 1.定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str)->str:
     """
      在客户数据库搜索信息

     Args:
            query: str,客户查询字符串，如："张三" 或 "李四"

     Returns:
            str: 客户记录字符串，包括客户姓名、等级、最近购买日期和累计消费
     """
     if "张三" in query.lower():
          return "客户记录: 张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
     elif "李四" in query.lower():
          return "客户记录: 李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
     else:
         return f"关于客户{query}，无记录"

@tool(parse_docstring=True)
def send_email(customer: str)->str:
    """
      发送感谢邮件

     Args:
            customer: str,客户名字，如："张三" 或 "李四"

     Returns:
            str: 确认消息，保护已发送的客户名称
     """
    return f"已向客户：{customer}发送感谢邮件"

# 2.初始化模型
model = ChatOpenAI(
    # model="carstenuhlig/omnicoder-9b:latest", # 在这里不好用
    model="qwen3-vl:latest", # ok
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
)

# 3.定义Pydantic结构类
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None,description="客户姓名")
    customer_tier: Literal["潜在客户","普通客户","VIP客户","流失风险"] = Field("潜在客户",
                                    description="客户只能是：潜在客户、普通客户、VIP客户和流失风险")
    recent_activity: str = Field(None,description="最近活动")
    spending_level: Literal["低","中","高"] = Field(None,description="消费水平")
    send_email: bool = Field(False,description="是否已发送感谢邮件")


# 4.定义agent
agent = create_agent(
    model=model,
    tools=[search_customer_database,send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis), # 结构化输出的第3种方式
    system_prompt=SystemMessage(content="""
    请分析知道客户的情况:
    1.先搜索客户数据库了解最新情况
    2.如果是VIP客户，则发送感谢邮件
    3.基于搜索结果生成结构化分析报告
    4.如果用户提问与客户记录无关或者找不到客户信息，
    则返回空对象，不发送感谢邮件。
    """)
)
# 3.调用
result = agent.invoke({
    "messages":[{"role":"user","content":"请分析客户张三"}]
})

rprint(result)






{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='73fad0b2-253f-458e-9e8b-da8ddbe27557'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 601,
                    'prompt_tokens': 497,
                    'total_tokens': 1098,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 278}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3-vl:latest',
                'system_fingerprint': 'fp_ollama',
                'id': 'chatcmpl-552',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a104bf-6681-7a30-9c88-913975f31066-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_hpfmdjlk',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 497,
                'output_tokens': 601,
                'total_tokens': 1098,
                'input_token_details': {'cache_read': 278},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='客户记录: 张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='f264b644-d78c-4265-bfe3-813b6ac8916a',
            tool_call_id='call_hpfmdjlk'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 276,
                    'prompt_tokens': 567,
                    'total_tokens': 843,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 495}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3-vl:latest',
                'system_fingerprint': 'fp_ollama',
                'id': 'chatcmpl-381',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a104c1-a183-7ab1-9b55-f7bcce347eea-0',
            tool_calls=[
                {'name': 'send_email', 'args': {'customer': '张三'}, 'id': 'call_s6irhf26', 'type': 'tool_call'}
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 567,
                'output_tokens': 276,
                'total_tokens': 843,
                'input_token_details': {'cache_read': 495},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='已向客户：张三发送感谢邮件',
            name='send_email',
            id='33914404-756a-42e8-ac18-d3a11850bf53',
            tool_call_id='call_s6irhf26'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 308,
                    'prompt_tokens': 610,
                    'total_tokens': 918,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 565}
                },
                'model_provider': 'openai',
                'model_name': 'qwen3-vl:latest',
                'system_fingerprint': 'fp_ollama',
                'id': 'chatcmpl-480',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--